# Emotion LoRA for the Helium motion student — RunPod edition

Trains an **emotion adapter for the new streaming pipeline** (`CausalMotionStudent` +
distilled renderer) — the counterpart of the old pipeline's `05_train_emotion_lora.ipynb`.

**Old vs new.** The old emotion stage conditioned the *FMGenerator* on wav2vec2 features
with an `EmotionConditioner` bias + LoRA on the FMT blocks. The new pipeline has no
FMGenerator and no wav2vec2: the avatar is driven by the distilled **Helium student**, a
38 M causal transformer that consumes PersonaPlex hidden states (4096-d @ 12.5 Hz) and
emits 2×32-D motion frames per 80 ms step. This notebook re-targets the same mechanism:

| old (FM pipeline) | new (this notebook) |
|---|---|
| wav2vec2 audio features (768-d, 25 fps) | Helium hidden states (4096-d, 12.5 Hz) extracted from the emotional audio |
| FMGenerator + EmotionConditioner (32-d conditioning bias) | CausalMotionStudent + StudentEmotionConditioner (512-d stream bias) |
| LoRA on `fmt.blocks` attn/mlp + `c_embedder` + decoder | LoRA on student blocks attn/mlp + output heads |
| full IMTRenderer | distilled single-identity renderer (`renderer_narrow.pt`) |
| style LoRA merged first | none needed — the base student already *is* the production model |

**Fully self-contained**: on a bare pod it clones the repo, builds the Python env, and
downloads all weights (gated base model + public avatar assets) — no other notebook needed.

**Steps**
0. one-time pod setup: GPU/disk preflight -> system libs -> clone
   `MoshiHead/speech2face_orin_n_original` -> `setup_env_x86.sh` (Python 3.10 env) ->
   `prepare_x86.sh` (~19 GB weights + GPU probe). All idempotent — re-runs skip finished
   steps (~40-70 min the first time, seconds afterwards)
1. validate the pod (repo, env, weights) and the dataset
2. **extract Helium** for every clip: the PersonaPlex trunk is teacher-forced on the
   *agent* stream with the clip's Mimi codes — the exact reply path the live server feeds
   the student (same engine class, same Robert_5 voice + persona prompt, same int4
   conversion as `run_x86.sh`)
3. train the conditioner + LoRA (everything else frozen; the Helium input path is never
   touched — the old pipeline's lip-sync guarantee)
4. validate the adapter and render test videos for all six emotions

**Prerequisites**
- a GPU pod (compute capability >= 8.0, >= 20 GiB VRAM) with >= 60 GB free on `/workspace`
- an **HF token** with the `nvidia/personaplex-7b-v1` license accepted (gated base model):
  paste it in the config cell, save it to `/workspace/hf_token.txt`, or set the `HF_TOKEN`
  env var — otherwise cell 0e prompts for it
- the emotion dataset on the pod: `motion/<stem>.pt` (`[T,32]`, from the old
  preprocessing) plus the **raw emotional wavs** with matching stems (the old
  `audio/*.npy` wav2vec2 features cannot be used — Mimi needs waveforms)

Nothing in the repo is modified; new files live under `src/niloys/emotion/`.

In [ ]:
import os, re, json, glob, shlex, subprocess, time, urllib.request
from pathlib import Path

# ---------------------------------------------------------------- paths (edit these)
REPO_DIR            = "/workspace/speech2face"
EMOTION_DATASET_DIR = "/workspace/emotion_dataset"     # motion/<stem>.pt from the OLD preprocessing
RAW_AUDIO_DIR       = os.path.join(EMOTION_DATASET_DIR, "wavs")   # raw emotional speech, matching stems
EXPERIMENT_NAME     = "emotion_lora_v1"

# ---- source repo for the one-time bootstrap (cell 0c clones it into REPO_DIR)
REPO_URL = "https://github.com/MoshiHead/speech2face_orin_n_original_emotion_adapter_v1.git"
BRANCH   = ""                       # "" = the repo's default branch

# ---- Hugging Face token (needed ONCE, by cell 0e, for the GATED base model
#      nvidia/personaplex-7b-v1 - accept the license on its HF page first):
#      paste "hf_..." here, or save it to /workspace/hf_token.txt, or set the
#      HF_TOKEN env var; if none is found, cell 0e prompts securely
token = "GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
HF_TOKEN = "hf_" + token

# ---- Python interpreter of the pod env (same resolution as run_x86.sh):
#      PY_OVERRIDE -> .python_path (written by prepare_x86.sh) -> env/bin/python -> python3
PY_OVERRIDE = ""                   # e.g. "/workspace/speech2face/env/bin/python"

def _resolve_python():
    pp = os.path.join(REPO_DIR, ".python_path")
    for cand in (PY_OVERRIDE,
                 open(pp).read().strip() if os.path.exists(pp) else "",
                 os.path.join(REPO_DIR, "env", "bin", "python")):
        if cand and os.path.exists(cand):
            return cand
    return "python3"

PY = _resolve_python()

def audit_base(include_live=False):
    # every base-setup artifact the emotion pipeline needs:
    # returns rows of (group, path, ok, repairable_in_notebook)
    W = os.path.join(REPO_DIR, "weights")
    rows = []
    def add(group, path, repairable):
        rows.append((group, path, os.path.exists(path), repairable))
    pp = os.path.join(REPO_DIR, ".snap_path")
    snap = open(pp).read().strip() if os.path.exists(pp) else ""
    add("base model", pp, False)
    if snap:
        add("base model", os.path.join(snap, "model.safetensors"), False)
    add("base model", os.path.join(W, "mimi.safetensors"), False)
    add("base model", os.path.join(W, "tokenizer_spm_32k_3.model"), False)
    st = os.path.join(W, "motionM.pt")
    if not os.path.exists(st):
        st = os.path.join(W, "motion_student_38M.pt")
    add("avatar assets", st, True)
    for f in ("renderer_narrow.pt", "silence_helium_mean.pt",
              "blink_motion.pt", "ref.jpeg"):
        add("avatar assets", os.path.join(W, f), True)
    add("avatar assets", os.path.join(W, "voices", VOICE), True)
    add("gpu probe", os.path.join(REPO_DIR, ".gpu_probe.env"), True)
    if include_live:
        for f in ("pd_quant_onpol_robert5_plaus2.pt", "mimi_dec_gan_robert5.pt"):
            add("robert5 gan", os.path.join(W, f), True)
    return rows

def print_audit(rows):
    print("%-14s %-8s %s" % ("group", "status", "artifact"))
    for group, path, ok, _ in rows:
        print("%-14s %-8s %s" % (group, "OK" if ok else "MISSING", path))

OUT_DIR    = os.path.join(EMOTION_DATASET_DIR, EXPERIMENT_NAME)
HELIUM_DIR = os.path.join(OUT_DIR, "helium")
CKPT_DIR   = os.path.join(OUT_DIR, "checkpoints")
VIDEO_DIR  = os.path.join(OUT_DIR, "final_emotion_tests")

# -------- PersonaPlex session for extraction (must match the live server / run_x86.sh)
VOICE       = "Robert_5.pt"
TEXT_PROMPT = ("You are Robert, an assistant from RB Labs. You have knowledge about crypto, "
               "finance, investment, technology, and general topics. Answer every question "
               "completely, descriptively, and with useful detail.")
EXTRACT_MAX_CLIPS    = 0          # 0 = all; set e.g. 60 for a quick smoke test
EXTRACT_RESET_MODE   = "emotion"  # re-apply the voice/persona prompt between emotion groups
EXTRACT_INT4         = "auto"     # "auto" = read .gpu_probe.env like run_x86.sh; "32" = int4; "0" = bf16
EXTRACT_WARMUP_STEPS = 8          # training windows never start closer than this to a clip start

# ---------------------------------------------------------------- training knobs
BASE_STUDENT = ""                 # "" = auto: weights/motionM.pt, else weights/motion_student_38M.pt
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 32, 64.0, 0.05
LR, COND_LR_MULT = 1e-4, 2.0
BATCH_SIZE      = 16
WINDOW_STEPS    = 64              # 64 x 80 ms = 5.12 s
EMOTION_DROPOUT = 0.1             # classifier-free (zeroed-bias) dropout
MAX_STEPS       = 20000
VAL_EVERY, SAVE_EVERY = 1000, 2000
TEST_CLIP         = ""            # stem for the final emotion videos; "" = first extracted clip
TEST_MAX_SECONDS  = 12.0

# -------- periodic test videos during training (the old pipeline's every-10-epochs
#          test, adapted to the new architecture - see the training section)
TEST_EVERY_N_EPOCHS = 10          # render a test video every N epochs (0 = off)
TEST_VIDEO_CLIP   = ""            # helium clip for the periodic test; "" = TEST_CLIP/first extracted
TEST_VIDEO_SECONDS = 6.0          # length of each periodic test video
TEST_EMOTION_ID   = -1            # -1 = cycle all six emotions across tests; or 0-5
MOTION_MAP        = "affine"      # affine | clamp | none - renderer range fix (see render section)

for d in (OUT_DIR, HELIUM_DIR, CKPT_DIR, VIDEO_DIR):
    os.makedirs(d, exist_ok=True)

def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (REPO_DIR if os.path.isdir(REPO_DIR) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

print("repo     :", REPO_DIR)
print("python  :", PY)
print("dataset  :", EMOTION_DATASET_DIR)
print("raw wavs :", RAW_AUDIO_DIR)
print("output   :", OUT_DIR)

## 0 · One-time pod setup (clone -> env -> weights)

Fully self-contained bootstrap: on a bare pod these five cells take you from nothing to a
complete base setup. **Every cell is idempotent** — re-running skips finished steps, so the
first run takes ~40-70 min (mostly downloads) and later runs take seconds.

| cell | what it does | first-run time |
|---|---|---|
| 0a | GPU / driver / disk preflight | seconds |
| 0b | system libraries (`apt`: libsndfile1, ffmpeg, …) | < 1 min |
| 0c | clone `MoshiHead/speech2face_orin_n_original` into `REPO_DIR` | < 1 min |
| 0d | build the Python 3.10 env (`setup_env_x86.sh`: torch cu128, pinned deps, TensorRT) | ~8-20 min |
| 0e | download **~19 GB** of weights (`prepare_x86.sh`: gated base model via your HF token + public avatar assets, sha256-checked) + GPU probe | ~15-40 min |

In [ ]:
# 0a - GPU / driver / disk preflight
import shutil
sh("nvidia-smi --query-gpu=gpu_name,driver_version,memory.total,compute_cap --format=csv",
   check=False)
u = shutil.disk_usage("/workspace")
print("disk /workspace: %.0f GB total, %.0f GB free" % (u.total / 2**30, u.free / 2**30))
if u.free < 60 * 2**30:
    print("!! under 60 GB free - the setup needs ~30 GB (env + ~19 GB weights + TRT engine)")

In [ ]:
# 0b - system libraries (no-op when already installed)
sh("DEBIAN_FRONTEND=noninteractive apt-get update -qq && "
   "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "libsndfile1 ffmpeg libgl1 libglib2.0-0 util-linux")
print("system libraries: OK")

In [ ]:
# 0c - clone the speech2face repo (skipped when already cloned)
need = [os.path.join(REPO_DIR, "src", "moshi_pkg", "moshi"),
        os.path.join(REPO_DIR, "setup_env_x86.sh"),
        os.path.join(REPO_DIR, "prepare_x86.sh"),
        os.path.join(REPO_DIR, "requirements.txt"),
        os.path.join(REPO_DIR, "tools", "gpu_probe.py")]
if all(os.path.exists(p) for p in need):
    print("repo already cloned - skipping")
else:
    os.makedirs(REPO_DIR, exist_ok=True)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print("repo has .git but files are missing - pulling latest ...")
        sh("git pull --ff-only", cwd=REPO_DIR)
    else:
        tmp = REPO_DIR.rstrip("/") + ".clone-tmp"
        print("cloning %s ..." % REPO_URL)
        sh("rm -rf %s && git clone --depth 1 %s%s %s"
           % (shlex.quote(tmp), ("-b " + shlex.quote(BRANCH) + " ") if BRANCH else "",
              shlex.quote(REPO_URL), shlex.quote(tmp)))
        # merge the clone into REPO_DIR without overwriting anything already there
        sh("cp -rn %s/. %s/ && rm -rf %s"
           % (shlex.quote(tmp), shlex.quote(REPO_DIR), shlex.quote(tmp)))
# always (re)apply the exec bit - the skip path above may follow a copy that lost it
sh("chmod +x setup_env_x86.sh prepare_x86.sh", cwd=REPO_DIR)
missing = [p for p in need if not os.path.exists(p)]
assert not missing, "clone incomplete - missing:\n  " + "\n  ".join(missing)
print("repo ready:", REPO_DIR)

In [ ]:
# 0d - Python 3.10 env: torch cu128, pinned deps, sphn + TensorRT (first run ~8-20 min)
if os.path.exists(os.path.join(REPO_DIR, "env", "bin", "python")):
    print("env already built - skipping (delete %s to rebuild)"
          % os.path.join(REPO_DIR, "env"))
else:
    t0 = time.time()
    sh("bash setup_env_x86.sh", cwd=REPO_DIR)
    print("env built in %.1f min" % ((time.time() - t0) / 60))
assert os.path.exists(os.path.join(REPO_DIR, "env", "bin", "python")), \
    "env build failed - scroll up for the failing step"

In [ ]:
# 0e - weights: gated base model (HF token) + public avatar assets, sha256-checked,
#      + GPU probe (first run ~15-40 min, ~19 GB)
if (os.path.exists(os.path.join(REPO_DIR, ".snap_path"))
        and os.path.isdir(os.path.join(REPO_DIR, "assets", "avatar_assets"))):
    print("weights already downloaded - skipping (delete weights/ + assets/ to re-download)")
else:
    if not HF_TOKEN:
        HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
    if not HF_TOKEN and os.path.exists("/workspace/hf_token.txt"):
        HF_TOKEN = open("/workspace/hf_token.txt").read().strip()
        print("HF token loaded from /workspace/hf_token.txt")
    if not HF_TOKEN:
        import getpass
        HF_TOKEN = getpass.getpass("Hugging Face READ token (input hidden): ").strip()
    assert HF_TOKEN.startswith("hf_"), \
        "token should start with 'hf_' - check you pasted a valid READ token"
    t0 = time.time()
    sh("bash prepare_x86.sh", cwd=REPO_DIR, env={"HF_TOKEN": HF_TOKEN})
    print("weights ready in %.1f min" % ((time.time() - t0) / 60))
# the env + .python_path now exist - (re)resolve the interpreter for all later cells
PY = _resolve_python()
print("python  :", PY)

## 1 · The shared emotion module

`StudentEmotionConditioner` (one-hot(7) → Linear → LayerNorm → SiLU, a 512-d bias added to
the student's token stream), the `EmotionMotionStudent` subclass (the live bias sits in a
buffer updated **in place**, so the captured CUDA graph keeps working when the emotion
switches mid-conversation), and the LoRA helpers. Written to
`src/niloys/emotion/emotion_student.py`; `emotion_live_streaming.ipynb` writes the
identical file.

In [ ]:
import os
CONTENT = '"""Emotion conditioning for the Helium motion student (training + live).\n\nFaithful port of the old FM pipeline\'s emotion mechanism\n(IMTalker generator/FM.py::EmotionConditioner + generator/train_lora.py)\nre-targeted at the distilled CausalMotionStudent (src/niloys/student/model.py):\n\n  * StudentEmotionConditioner  one-hot(7) -> Linear(7 -> d_model) -> LayerNorm\n                               -> SiLU, added to the student\'s embedded token\n                               stream. 10% of training steps get a zeroed bias\n                               (classifier-free dropout), like the old pipeline.\n  * EmotionMotionStudent       CausalMotionStudent subclass. The live bias\n                               lives in a registered buffer updated IN PLACE by\n                               set_emotion(), so the captured CUDA graph keeps\n                               reading the same storage when the emotion is\n                               switched mid-conversation.\n  * LoRA helpers               adapters on the transformer blocks (attn.qkv,\n                               attn.proj, mlp.0, mlp.3) and the output heads\n                               (head_abs, head_res, head_gate). The Helium input\n                               path (helium_ln / helium_proj / prev_enc /\n                               fuse_*) is never touched - the same lip-sync\n                               guarantee as freezing the old audio projections.\n\nOnly NEW files live under src/niloys/emotion/ - nothing existing is modified.\n"""\nfrom __future__ import annotations\n\nimport re\n\nimport torch\nimport torch.nn as nn\n\nfrom student.model import CausalMotionStudent, StudentConfig\n\nEMOTION_MAP = {\n    "neutral": 0, "calm": 0,\n    "happy": 1, "happiness": 1, "joy": 1, "amused": 1,\n    "sad": 2, "sadness": 2,\n    "angry": 3, "anger": 3, "rage": 3,\n    "surprise": 4, "surprised": 4,\n    "fear": 5, "fearful": 5, "afraid": 5,\n}\nEMOTION_NAMES = ["neutral", "happy", "sad", "angry", "surprise", "fear"]\nNUM_EMOTIONS = 6\nNUM_CLASSES = NUM_EMOTIONS + 1      # one-hot width incl. the classifier-free null slot\nADAPTER_FORMAT = "student_emotion_adapter_v1"\n\n\ndef parse_emotion_from_stem(stem: str):\n    """MEAD-style names: M003_happy_level_1_001, W011_sad_3_012, ..."""\n    for part in re.split(r"[_\\-\\.\\s]+", stem.lower()):\n        if part in EMOTION_MAP:\n            return EMOTION_MAP[part]\n    return None\n\n\nclass StudentEmotionConditioner(nn.Module):\n    """emotion id -> [B,T,d_model] additive bias (the old EmotionConditioner at d_model=512)."""\n\n    def __init__(self, d_model: int, dropout: float = 0.1):\n        super().__init__()\n        self.dropout = float(dropout)\n        self.emotion_embedding = nn.Embedding(NUM_CLASSES, NUM_CLASSES)\n        with torch.no_grad():\n            self.emotion_embedding.weight.copy_(torch.eye(NUM_CLASSES))\n        self.emotion_embedding.weight.requires_grad_(False)\n        self.emotion_projection = nn.Linear(NUM_CLASSES, d_model)\n        self.layer_norm = nn.LayerNorm(d_model)\n        self.activation = nn.SiLU()\n\n    def forward(self, emotion_ids, batch_size: int, seq_len: int, train: bool = False):\n        emb = self.emotion_embedding(emotion_ids)           # [B,7] one-hot\n        emb = self.emotion_projection(emb)                  # [B,C]\n        emb = self.layer_norm(emb)\n        emb = self.activation(emb)\n        emb = emb.unsqueeze(1).expand(batch_size, seq_len, emb.shape[-1])\n        if train and self.dropout > 0:\n            mask = torch.rand(batch_size, 1, 1, device=emb.device) < self.dropout\n            emb = torch.where(mask, torch.zeros_like(emb), emb)\n        return emb\n\n    @torch.no_grad()\n    def bias_for(self, idx: int, device, dtype):\n        ids = torch.tensor([int(idx)], device=device, dtype=torch.long)\n        return self.forward(ids, 1, 1, train=False).to(device=device, dtype=dtype)\n\nclass EmotionMotionStudent(CausalMotionStudent):\n    """CausalMotionStudent + emotion bias added after _embed()."""\n\n    def __init__(self, cfg: StudentConfig, emotion_dropout: float = 0.1):\n        super().__init__(cfg)\n        self.emotion_conditioner = StudentEmotionConditioner(cfg.d_model, emotion_dropout)\n        self.register_buffer("emotion_bias", torch.zeros(1, 1, cfg.d_model),\n                             persistent=False)\n        self._emotion_bias = None       # streamed bias (the buffer) while live\n\n    # -- live --------------------------------------------------------------\n    @torch.no_grad()\n    def set_emotion(self, idx: int):\n        emb = self.emotion_conditioner.bias_for(\n            idx, self.emotion_bias.device, self.emotion_bias.dtype)\n        self.emotion_bias.copy_(emb)    # in place: the captured CUDA graph keeps working\n        self._emotion_bias = self.emotion_bias\n\n    # -- shared ------------------------------------------------------------\n    def _embed(self, helium, prev_motion):\n        x = super()._embed(helium, prev_motion)\n        if self._emotion_bias is not None:\n            x = x + self._emotion_bias\n        return x\n\n    # -- training ----------------------------------------------------------\n    def forward_with_emotion(self, helium, prev_motion, emotion_ids, train: bool = True):\n        B, T = helium.shape[0], helium.shape[1]\n        bias = self.emotion_conditioner(emotion_ids, B, T, train=train)\n        prev = self._emotion_bias\n        self._emotion_bias = bias\n        try:\n            return super().forward(helium, prev_motion)\n        finally:\n            self._emotion_bias = prev\n\n\n# ---------------------------------------------------------------- LoRA\nLORA_TARGETS = ("attn.qkv", "attn.proj", "mlp.0", "mlp.3",\n                "head_abs", "head_res", "head_gate")\n\n\ndef apply_emotion_lora(model, rank: int = 32, alpha: float = 64.0, dropout: float = 0.05):\n    """Freeze everything, unfreeze the conditioner, wrap LoRA around the blocks\'\n    attention/MLP projections and the output heads. Returns #wrapped layers."""\n    from generator.train_lora import LoRALinear      # needs src/imtalker on sys.path\n    for p in model.parameters():\n        p.requires_grad_(False)\n    for p in model.emotion_conditioner.parameters():\n        p.requires_grad_(True)\n    model.emotion_conditioner.emotion_embedding.weight.requires_grad_(False)\n    n = 0\n    for name, mod in list(model.named_modules()):\n        if not isinstance(mod, nn.Linear):\n            continue\n        if not any(name == t or name.endswith("." + t) for t in LORA_TARGETS):\n            continue\n        parent_name, _, attr = name.rpartition(".")\n        parent = model.get_submodule(parent_name) if parent_name else model\n        setattr(parent, attr, LoRALinear(mod, rank=rank, alpha=alpha, dropout=dropout))\n        n += 1\n    return n\n\n\ndef save_emotion_adapter(model, path, meta):\n    from generator.train_lora import LoRALinear\n    sd = {"format": ADAPTER_FORMAT, "meta": meta,\n          "conditioner": {k: v.detach().cpu()\n                          for k, v in model.emotion_conditioner.state_dict().items()},\n          "lora": {}}\n    for name, mod in model.named_modules():\n        if isinstance(mod, LoRALinear):\n            sd["lora"][name] = {"A": mod.lora_A.weight.detach().cpu(),\n                                "B": mod.lora_B.weight.detach().cpu()}\n    torch.save(sd, path)\n    return sd\n\n\ndef load_base_student(base_ckpt: str):\n    """Fresh trainable copy of the base student (weights + stats buffers ride along)."""\n    pack = torch.load(base_ckpt, map_location="cpu", weights_only=False)\n    cfg = StudentConfig(**pack["cfg"])\n    model = EmotionMotionStudent(cfg)\n    model.load_state_dict(pack["model"], strict=False)   # conditioner keys are new\n    return model, cfg, pack\n\n\ndef build_emotion_student(base_ckpt: str, adapter_ckpt: str, device,\n                          emotion_dropout: float = 0.0):\n    """Base student + merged emotion LoRA + loaded conditioner (eval, frozen).\n\n    The LoRA deltas are merged straight into the base Linear weights - the live\n    path keeps no LoRA wrappers, so the student\'s CUDA graph is unaffected.\n    """\n    model, cfg, _pack = load_base_student(base_ckpt)\n    ad = torch.load(adapter_ckpt, map_location="cpu", weights_only=False)\n    assert ad.get("format") == ADAPTER_FORMAT, \\\n        "not a student emotion adapter: %s" % adapter_ckpt\n    meta = ad.get("meta", {})\n    scale = float(meta.get("lora_alpha", 64.0)) / float(meta.get("lora_rank", 32))\n    for name, d in ad["lora"].items():\n        mod = model.get_submodule(name)\n        mod.weight.data.add_(scale * (d["B"] @ d["A"]).to(mod.weight.dtype))\n    model.emotion_conditioner.load_state_dict(ad["conditioner"])\n    model.emotion_conditioner.dropout = float(emotion_dropout)\n    model.to(device).float().eval()\n    for p in model.parameters():\n        p.requires_grad_(False)\n    return model, cfg, meta\n'
path = os.path.join(REPO_DIR, "src", "niloys", "emotion", "emotion_student.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))

## 2 · Environment and dataset validation

Checks the repo/env/weights from the base setup, the GPU, and the dataset: every
`motion/<stem>.pt` needs a raw audio file with the same stem and a parseable emotion label
(MEAD naming: `M003_happy_level_1_001`, `W011_sad_3_012`, …). The per-emotion counts tell
you whether the label parser matches your stems before anything is extracted.

If the audit flags **avatar assets** / **gpu probe** items as missing, run the repair
cell (2b) - it re-downloads the public ~250 MB assets from the same sha256-checked
source `prepare_x86.sh` uses (no token needed). Base-model items require cell 0e above
(HF token, ~16 GB).

In [ ]:
assert os.path.isdir(REPO_DIR), "repo missing - run the Section 0 setup cells above (0c clones it)"
assert os.path.exists(PY) or subprocess.run(
    ["bash", "-c", "command -v %s" % shlex.quote(PY)],
    capture_output=True).returncode == 0, (
    "python interpreter not found: %r - run cell 0d (env build) above, "
    "or set PY_OVERRIDE in the config cell" % PY)
sh("%s -c 'import sys, torch; print(\"python:\", sys.executable, \"| torch:\", torch.__version__)'"
   % shlex.quote(PY))

# ---------- base-setup audit ----------
rows = audit_base()
print_audit(rows)
fatal = [r for r in rows if not r[2] and not r[3]]
repair = [r for r in rows if not r[2] and r[3]]
if fatal or repair:
    sh("ls -la %s %s" % (shlex.quote(os.path.join(REPO_DIR, "weights")),
                         shlex.quote(os.path.join(REPO_DIR, "assets"))), check=False)
assert not fatal, (
    "base-model artifacts missing - run cell 0e above (HF token + ~16 GB "
    "base model): " + ", ".join(r[1] for r in fatal))
assert not repair, (
    "run the NEXT cell (repair): it downloads the public avatar assets "
    "(~250 MB) and probes the GPU. Or set BASE_STUDENT in the config cell "
    "if you have your own student checkpoint. Missing: "
    + ", ".join(os.path.basename(r[1]) for r in repair))

if not BASE_STUDENT:
    for cand in ("motionM.pt", "motion_student_38M.pt"):
        if os.path.exists(os.path.join(REPO_DIR, "weights", cand)):
            BASE_STUDENT = os.path.join(REPO_DIR, "weights", cand)
            break
assert BASE_STUDENT and os.path.exists(BASE_STUDENT), "no base student checkpoint in weights/"
print("base student:", BASE_STUDENT)

sh("nvidia-smi --query-gpu=name,compute_cap,memory.total,memory.used --format=csv,noheader",
   check=False)
out = subprocess.run(["bash", "-c",
                      "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if out and float(out.splitlines()[0]) / 1024 < 20:
    print("!! under 20 GiB VRAM - the int4 trunk alone needs ~14 GB; close other GPU processes")


# ---------- dataset ----------
motion_files = sorted(glob.glob(os.path.join(EMOTION_DATASET_DIR, "motion", "*.pt")))
audio = {}
for ext in ("wav", "mp3", "flac", "ogg", "m4a"):
    for f in glob.glob(os.path.join(RAW_AUDIO_DIR, "**", "*." + ext), recursive=True):
        audio.setdefault(Path(f).stem, f)
assert motion_files, "no motion/*.pt under %s" % EMOTION_DATASET_DIR
assert audio, "no raw audio under %s" % RAW_AUDIO_DIR

import collections
_PREVIEW = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
def _parse(stem):
    for part in re.split(r"[_\-\.\s]+", stem.lower()):
        if part in _PREVIEW:
            return _PREVIEW[part]
    return None
matched = [(Path(m).stem, _parse(Path(m).stem)) for m in motion_files
           if Path(m).stem in audio]
labeled = [(s, e) for s, e in matched if e is not None]
cnt = collections.Counter(e for _, e in labeled)
names = ["neutral", "happy", "sad", "angry", "surprise", "fear"]
print("motion clips: %d | with raw audio: %d | with emotion label: %d"
      % (len(motion_files), len(matched), len(labeled)))
print("per emotion :", {names[e]: cnt.get(e, 0) for e in range(6)})
print("examples    :", [s for s, _ in labeled[:5]])
assert labeled, "no clip has both raw audio and a parseable emotion label - check the stems"

## 2b - Repair - only if the audit flagged avatar-assets / gpu-probe items

Re-downloads the **public** avatar assets from `niloy629/personaplex-parallel-depformer`
(sha256-checked against `SHA256SUMS` - the same source and logic as `prepare_x86.sh`
step 4/6), links them into `weights/`, and runs the GPU probe. The gated base-model
artifacts (~16 GB) are **not** downloaded here - run cell 0e for those (HF token).
Idempotent: present files are skipped; downloads reuse the base setup's `hf/`
cache, so already-cached files are instant. Re-run the validation cell afterwards.

In [ ]:
rows = audit_base()
fatal = [r for r in rows if not r[2] and not r[3]]
missing = [r for r in rows if not r[2] and r[3]]
assert not fatal, ("base-model artifacts missing - run cell 0e above (HF token) "
                   "first: " + ", ".join(r[1] for r in fatal))
if not missing:
    print("nothing to repair - all base artifacts present")
else:
    print("repairing:", ", ".join(os.path.basename(r[1]) for r in missing))
    FETCH = ("import hashlib, os, sys\n"
             "from huggingface_hub import hf_hub_download\n"
             "REPO_ID = 'niloy629/personaplex-parallel-depformer'\n"
             "folder, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]\n"
             "want = {}\n"
             "for line in open(hf_hub_download(REPO_ID, folder + '/SHA256SUMS')):\n"
             "    h, f = line.split(); want[os.path.basename(f)] = h\n"
             "os.makedirs(dest, exist_ok=True)\n"
             "for f in files:\n"
             "    src = hf_hub_download(REPO_ID, folder + '/' + f)\n"
             "    h = hashlib.sha256()\n"
             "    with open(src, 'rb') as fh:\n"
             "        for chunk in iter(lambda: fh.read(1 << 24), bytes()):\n"
             "            h.update(chunk)\n"
             "    if h.hexdigest() != want[f]:\n"
             "        raise SystemExit('hash mismatch for ' + f)\n"
             "    dst = os.path.join(dest, f)\n"
             "    if os.path.islink(dst) or os.path.exists(dst):\n"
             "        os.remove(dst)\n"
             "    os.symlink(src, dst)\n"
             "    print('  fetched %s -> %s' % (f, src), flush=True)\n")
    fp = os.path.join(REPO_DIR, "_fetch_assets.py")
    open(fp, "w").write(FETCH)
    W = os.path.join(REPO_DIR, "weights")
    ASSETS = os.path.join(REPO_DIR, "assets")
    HF_ENV = {"HF_HOME": os.path.join(REPO_DIR, "hf"),
              "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    AV = ["motion_student_38M.pt", "renderer_narrow.pt", "silence_helium_mean.pt",
          "blink_motion.pt", "ref.jpeg", "Robert_5.pt"]
    def _have(f):
        p = os.path.join(W, "voices", f) if f == "Robert_5.pt" else os.path.join(W, f)
        return os.path.exists(p)
    need = [f for f in AV if not _have(f)]
    if not os.path.exists(os.path.join(ASSETS, "seanet_w12.onnx")):
        need.append("seanet_w12.onnx")
    if need:
        sh("%s %s avatar_assets %s %s"
           % (shlex.quote(PY), shlex.quote(fp), shlex.quote(ASSETS), " ".join(need)),
           env=HF_ENV)
        for f in need:
            if f == "seanet_w12.onnx":
                continue
            dst = os.path.join(W, "voices", f) if f == "Robert_5.pt" else os.path.join(W, f)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.islink(dst) or os.path.exists(dst):
                os.remove(dst)
            os.symlink(os.path.join(ASSETS, f), dst)
            print("linked %s" % dst)
    if not os.path.exists(os.path.join(REPO_DIR, ".gpu_probe.env")):
        sh("%s tools/gpu_probe.py .gpu_probe.env" % shlex.quote(PY))
    os.remove(fp)
    rows = audit_base()
    print_audit(rows)
    left = [r for r in rows if not r[2]]
    assert not left, "still missing after repair: " + ", ".join(r[1] for r in left)
    print("repair complete - all base artifacts present")

## 3 · Helium extraction

For every clip the PersonaPlex trunk is stepped at 12.5 Hz with **silence on the user
stream and the clip's Mimi codes teacher-forced on the agent stream**
(`moshi_tokens` in `LMGen.prepare_step_input`) — the exact reply path the live server
feeds the student, including the Robert_5 voice prompt, the persona text prompt, and the
int4 trunk conversion from `run_x86.sh`. The captured tensor is the same
`layer_hidden[:1,-1:]` the live server hands to `_sample_motion_from_helium`.

Clips are processed grouped by emotion so the voice/persona prompt is re-applied only when
the emotion changes (`EXTRACT_RESET_MODE="emotion"`; `"clip"` is cleaner but ~2x slower,
`"never"` fastest). The script is resumable (existing `.pt` files are skipped) and writes
a `manifest.json`. Expect roughly 2–10 s per clip depending on the GPU and prompt length.

In [ ]:
import os
CONTENT = '"""Extract per-80ms Helium hidden states for the emotion dataset by\nteacher-forcing the PersonaPlex trunk\'s AGENT stream with the emotional\nreference audio (moshi_tokens in LMGen.prepare_step_input).\n\nThis is the exact reply path the live server feeds the motion student from:\nsame engine class (MoshiOnlyEngineWithHidden), same Robert_5 voice + persona\ntext prompt, same int4 trunk conversion, same graphed _step with the layer\nhidden capture. The USER stream is held at silence, matching the live\nlistening state while the agent speaks.\n\nResumable: clips whose output .pt already exists are skipped.\n"""\nimport argparse, glob, json, os, sys, time\nfrom pathlib import Path\n\nAUDIO_EXTS = ("wav", "mp3", "flac", "ogg", "m4a")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--repo_root", required=True)\n    ap.add_argument("--motion_dir", required=True)\n    ap.add_argument("--raw_audio_dir", required=True)\n    ap.add_argument("--out_dir", required=True)\n    ap.add_argument("--voice", default="Robert_5.pt")\n    ap.add_argument("--text_prompt", required=True)\n    ap.add_argument("--max_clips", type=int, default=0)\n    ap.add_argument("--reset_mode", default="emotion",\n                    choices=["emotion", "clip", "never"])\n    ap.add_argument("--int4_g", type=int, default=32, help="0 = bf16 trunk")\n    args = ap.parse_args()\n\n    repo = Path(args.repo_root).resolve()\n    src = repo / "src"\n    os.environ.setdefault("IMTALKER_DIR", str(src / "imtalker"))\n    os.environ.setdefault("DESIGN_A_DIR", str(src / "niloys"))\n    for p in (str(src / "imtalker"), str(src / "moshi_pkg"), str(src / "niloys"),\n              str(src / "niloys" / "live"), str(src / "niloys" / "emotion")):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n\n    weights = repo / "weights"\n    snap = Path((repo / ".snap_path").read_text().strip())\n    moshi_weight = snap / "model.safetensors"\n    mimi_weight = weights / "mimi.safetensors"\n    tokenizer = weights / "tokenizer_spm_32k_3.model"\n    for f in (moshi_weight, mimi_weight, tokenizer):\n        assert f.exists(), "missing %s - run the Section 0 setup cells (or prepare_x86.sh) first" % f\n\n    import torch\n\n    if args.int4_g > 0:\n        # identical conversion to the live server (the try19 int4 loader hook)\n        from moshi.models import loaders as _loaders\n        from int4_linear import convert_bf16 as _conv_int4\n        _orig_get_lm = _loaders.get_moshi_lm\n\n        def _get_moshi_lm_int4(*a, **k):\n            lm = _orig_get_lm(*a, **k)\n            n = _conv_int4(lm.transformer, args.int4_g)\n            torch.cuda.empty_cache()\n            print("[extract] int4 G=%d: %d Linears converted" % (args.int4_g, n),\n                  flush=True)\n            return lm\n        _loaders.get_moshi_lm = _get_moshi_lm_int4\n        print("[extract] int4 loader hook installed (matches the live server)",\n              flush=True)\n    else:\n        print("[extract] bf16 trunk", flush=True)\n\n    from imtalker_native_passthrough import (\n        MoshiOnlyEngineWithHidden, load_audio_24k)\n    from emotion_student import EMOTION_NAMES, parse_emotion_from_stem\n\n    engine = MoshiOnlyEngineWithHidden(\n        moshi_root=str(weights),\n        mimi_hf_repo="nvidia/personaplex-7b-v1",\n        device="cuda",\n        cfg_coef=1.0,\n        placeholder_jpeg_b64="",\n        moshi_weight=str(moshi_weight),\n        mimi_weight=str(mimi_weight),\n        tokenizer=str(tokenizer),\n        quantize_4bit=False,\n        num_codebooks=8,\n        context=None,\n        voice_prompt=args.voice,\n        voice_prompt_dir=str(weights / "voices"),\n        text_prompt=args.text_prompt,\n    )\n    mimi = engine.mimi\n    FRAME = int(mimi.sample_rate / mimi.frame_rate)     # 1920 samples = 80 ms\n    device = engine.device\n\n    # Full-clip bulk encodes hit the encoder transformer with lengths its CUDA\n    # graph was never captured for (CUDAGraphed is shape-static) - bypass it,\n    # exactly like the live server\'s raw voice prompt path. Re-applied after\n    # every streaming reset in case the state (and its graph) is re-created.\n    def _disable_mimi_enc_graph():\n        st = getattr(mimi, "_streaming_state", None)\n        g = getattr(st, "graphed_tr_enc", None)\n        if g is not None:\n            g.disable = True\n\n    # one fixed silence code column for the USER stream (encoded once, reused)\n    mimi.reset_streaming()\n    _disable_mimi_enc_graph()\n    with torch.no_grad():\n        silence = mimi.encode(torch.zeros(1, 1, FRAME, device=device))[:, :8].clone()\n\n    # ---- clip list: motion stems that also have raw audio + an emotion label\n    motion_files = sorted(glob.glob(os.path.join(args.motion_dir, "*.pt")))\n    audio = {}\n    for ext in AUDIO_EXTS:\n        for f in glob.glob(os.path.join(args.raw_audio_dir, "**", "*." + ext),\n                           recursive=True):\n            audio.setdefault(Path(f).stem, f)\n    clips, n_no_audio, n_no_emo = [], 0, 0\n    for mf in motion_files:\n        stem = Path(mf).stem\n        eid = parse_emotion_from_stem(stem)\n        if stem not in audio:\n            n_no_audio += 1\n            continue\n        if eid is None:\n            n_no_emo += 1\n            continue\n        clips.append((stem, eid, mf, audio[stem]))\n    clips.sort(key=lambda c: (c[1], c[0]))   # group by emotion: fewer voice-prompt resets\n    if args.max_clips:\n        clips = clips[: args.max_clips]\n    print("[extract] %d clips (%d skipped: no audio, %d skipped: no emotion label)"\n          % (len(clips), n_no_audio, n_no_emo), flush=True)\n\n    os.makedirs(args.out_dir, exist_ok=True)\n    manifest = {"clips": [], "frame_samples": FRAME, "sample_rate": 24000,\n                "voice": args.voice, "text_prompt": args.text_prompt,\n                "int4_g": args.int4_g, "reset_mode": args.reset_mode,\n                "created": time.strftime("%Y-%m-%d %H:%M:%S")}\n    last_emotion, t_start = None, time.time()\n    for i, (stem, eid, mf, af) in enumerate(clips):\n        out_path = os.path.join(args.out_dir, stem + ".pt")\n        if os.path.exists(out_path):\n            continue                                    # resumable\n        wav_np = load_audio_24k(af)                     # float32 mono @ 24 kHz\n        motion = torch.load(mf, map_location="cpu", weights_only=False)\n        if isinstance(motion, dict):\n            motion = motion.get("motion", motion.get("motion_latents"))\n        n_steps = min(int(len(wav_np)) // FRAME, int(motion.shape[0]) // 2)\n        if n_steps < 4:\n            print("[extract] skip %s: too short" % stem, flush=True)\n            continue\n        wav = torch.from_numpy(wav_np[: n_steps * FRAME].copy()).view(1, 1, -1).to(device)\n        mimi.reset_streaming()\n        _disable_mimi_enc_graph()\n        with torch.no_grad():\n            codes = mimi.encode(wav)[:, :8]             # [1,8,T] agent-stream codes\n        if (args.reset_mode == "clip"\n                or (args.reset_mode == "emotion" and eid != last_emotion)):\n            engine.reset_session()                      # re-apply Robert_5 + persona\n        last_emotion = eid\n        hidden = []\n        with torch.no_grad():\n            for t in range(n_steps):\n                out = engine.lm_gen._step(input_tokens=silence,\n                                          moshi_tokens=codes[:, :, t:t + 1])\n                if out is None:                         # first call after a reset\n                    continue\n                hidden.append(out[2][:1, -1:].detach().float().cpu())\n        h = torch.cat(hidden, 1)[0].contiguous()        # [T,4096]\n        torch.save({"helium": h, "emotion_id": eid, "stem": stem}, out_path)\n        manifest["clips"].append({"stem": stem, "emotion_id": eid,\n                                  "emotion": EMOTION_NAMES[eid],\n                                  "steps": int(h.shape[0])})\n        el = time.time() - t_start\n        print("[extract] %4d/%4d %-42s %-9s T=%4d  %5.1fs/clip  elapsed %5.1f min"\n              % (i + 1, len(clips), stem, EMOTION_NAMES[eid], h.shape[0],\n                 el / (i + 1), el / 60), flush=True)\n    with open(os.path.join(args.out_dir, "manifest.json"), "w") as f:\n        json.dump(manifest, f, indent=1)\n    print("EXTRACTION DONE: %d clips -> %s" % (len(manifest["clips"]), args.out_dir),\n          flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
path = os.path.join(REPO_DIR, "src", "niloys", "emotion", "extract_helium.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))

In [ ]:
probe = os.path.join(REPO_DIR, ".gpu_probe.env")
int4_g = EXTRACT_INT4
if int4_g == "auto":
    int4_g = "32"
    if os.path.exists(probe):
        m = re.search(r"TRY19_INT4_G=(\d+)", open(probe).read())
        if m:
            int4_g = m.group(1)
print("extraction trunk: %s" % ("int4 G=" + int4_g if int4_g != "0" else "bf16"))

EXTRACT_LOG = os.path.join(OUT_DIR, "extract.log")
sh("pkill -f extract_helium.py; sleep 1", check=False)
cmd = ("%s -u src/niloys/emotion/extract_helium.py "
       "--repo_root . --motion_dir %s --raw_audio_dir %s --out_dir %s "
       "--voice %s --text_prompt %s --max_clips %d --reset_mode %s --int4_g %s"
       % (shlex.quote(PY),
          shlex.quote(os.path.join(EMOTION_DATASET_DIR, "motion")),
          shlex.quote(RAW_AUDIO_DIR), shlex.quote(HELIUM_DIR),
          shlex.quote(VOICE), shlex.quote(TEXT_PROMPT),
          EXTRACT_MAX_CLIPS, EXTRACT_RESET_MODE, int4_g))
sh("nohup %s > %s 2>&1 & sleep 3; pgrep -af extract_helium.py | head -2"
   % (cmd, shlex.quote(EXTRACT_LOG)))

In [ ]:
# follow the extraction (re-runnable); returns when the process exits
t0 = time.time()
while True:
    alive = subprocess.run(["pgrep", "-f", "extract_helium.py"],
                           capture_output=True).returncode == 0
    tail = subprocess.run(["bash", "-c", "tail -n 2 %s 2>/dev/null" % EXTRACT_LOG],
                          capture_output=True, text=True).stdout.strip().replace("\n", "  |  ")
    print("\r%7.0fs  %s" % (time.time() - t0, tail[-160:]), end="", flush=True)
    if not alive:
        break
    time.sleep(15)
print()
sh("tail -n 6 %s" % EXTRACT_LOG, check=False)

In [ ]:
# sanity: the extracted Helium statistics should match the stats the student was
# trained with (a wrong layer / wrong stream would show up here immediately)
VERIFY = r"""
import glob, sys
import torch
files = sorted(glob.glob(r'HELIUM_DIR_X/*.pt'))
print('extracted clips:', len(files))
assert files, 'nothing extracted - see the extraction log'
pack = torch.load(r'BASE_STUDENT_X', map_location='cpu', weights_only=False)
hm, hs = pack['model']['helium_mean'], pack['model']['helium_std']
sample = files[:: max(1, len(files) // 40)][:40]
Hs, steps = [], 0
for f in sample:
    d = torch.load(f, map_location='cpu', weights_only=False)
    h = d['helium'] if isinstance(d, dict) else d
    steps += h.shape[0]
    Hs.append(h[:: max(1, h.shape[0] // 50)])
H = torch.cat(Hs, 0)
mu, sd = H.mean(0), H.std(0).clamp_min(1e-6)
print('steps sampled :', steps, ' helium frame shape:', tuple(h.shape))
print('|cos(extracted mean, student helium_mean)|: %.4f'
      % float(torch.nn.functional.cosine_similarity(mu, hm, dim=0).abs()))
print('std ratio (extracted / student): mean %.3f median %.3f'
      % (float((sd / hs).mean()), float((sd / hs).median())))
print('sane ranges: cos > 0.5 and ratio within 0.5..2')
"""
verify = (VERIFY
          .replace("HELIUM_DIR_X", HELIUM_DIR)
          .replace("BASE_STUDENT_X", BASE_STUDENT))
open(os.path.join(REPO_DIR, "_verify_extract.py"), "w").write(verify)
sh("%s _verify_extract.py && rm _verify_extract.py" % shlex.quote(PY))

## 4 · Training

Plain PyTorch (the pod env has no Lightning). What trains and what does not — the same
contract as the old pipeline:

| component | status |
|---|---|
| `StudentEmotionConditioner` (projection + LN) | **trained** (lr × 2) |
| LoRA on `blocks.*.attn.qkv/proj`, `blocks.*.mlp.0/3`, `head_abs/head_res/head_gate` | **trained** (lr × 1) |
| Helium input path (`helium_ln`, `helium_proj`, `prev_enc`, `fuse_*`) + blocks' LayerNorms | frozen — preserves lip-sync |
| the one-hot `emotion_embedding` (identity matrix) | frozen, as in the old pipeline |

The objective is teacher-forced MSE on normalized motion over `WINDOW_STEPS`-long windows
with the student's `cfg.lookahead` label shift (output for step *s* pairs with
`helium[s+1]` — the documented 80 ms video lag). 10 % of steps get a zeroed emotion bias
(classifier-free dropout). Validation reports teacher-forced MSE per emotion, a
**free-running rollout NMSE** (one val clip per emotion — the number that predicts live
behaviour), and the learned emotion-bias geometry (norms + pairwise cosines).

**Periodic test videos — the old pipeline's every-10-epochs test, adapted.** The old
Lightning trainer had `EpochTestVideoCallback`: every N epochs, reference image + audio +
emotion → generator → renderer → `test_videos/`. The architectural translation here:
the *reference image* is **baked into the distilled renderer** (single-identity Robert;
no ref image needed at render time), and the *audio features* are the **pre-extracted
helium** of one fixed test clip — the 7 B trunk never runs inside the training process,
so testing cannot destabilize training. Every `TEST_EVERY_N_EPOCHS` epochs (an epoch =
one pass over the clip list, `ceil(len(train_ds)/batch_size)` steps) the trainer runs the
same rollout+render path as the final test-videos cell and writes
`emotion_output/test_videos/epoch_NNNN_<emotion>.mp4` (cycling through the six emotions;
`TEST_EMOTION_ID=-1`), plus an **epoch-0000 baseline** with the untrained adapter before
training starts. Rendering is `no_grad` + `eval()`/`train()` bracketed and never touches
the optimizer or the emotion bias.

In [ ]:
import os
CONTENT = '"""Train the emotion conditioner + LoRA adapters on the Helium motion student.\n\nPlain PyTorch (no Lightning in this env). Objective: teacher-forced MSE on\nnormalized motion over sliding windows, with the cfg.lookahead label shift the\nstudent was distilled with, and 10% classifier-free emotion dropout - the same\ntraining signal shape as the old FM pipeline\'s emotion stage.\n"""\nimport argparse, glob, json, os, random, sys, time\nfrom pathlib import Path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--repo_root", required=True)\n    ap.add_argument("--helium_dir", required=True)\n    ap.add_argument("--motion_dir", required=True)\n    ap.add_argument("--out_dir", required=True)\n    ap.add_argument("--base_student", required=True)\n    ap.add_argument("--lora_rank", type=int, default=32)\n    ap.add_argument("--lora_alpha", type=float, default=64.0)\n    ap.add_argument("--lora_dropout", type=float, default=0.05)\n    ap.add_argument("--lr", type=float, default=1e-4)\n    ap.add_argument("--cond_lr_mult", type=float, default=2.0)\n    ap.add_argument("--batch_size", type=int, default=16)\n    ap.add_argument("--window_steps", type=int, default=64)\n    ap.add_argument("--emotion_dropout", type=float, default=0.1)\n    ap.add_argument("--warmup_steps", type=int, default=8)\n    ap.add_argument("--max_steps", type=int, default=20000)\n    ap.add_argument("--val_every", type=int, default=1000)\n    ap.add_argument("--save_every", type=int, default=2000)\n    ap.add_argument("--grad_clip", type=float, default=1.0)\n    ap.add_argument("--weight_decay", type=float, default=0.01)\n    ap.add_argument("--val_split", type=float, default=0.05)\n    ap.add_argument("--seed", type=int, default=1234)\n    # periodic test videos (the old pipeline\'s every-10-epochs test, adapted to\n    # the new architecture: the ref image is baked into the distilled renderer,\n    # and the audio features are the PRE-EXTRACTED helium of a fixed clip - the\n    # 7B trunk never runs inside the training process)\n    ap.add_argument("--test_every_n_epochs", type=int, default=10, help="0 = off")\n    ap.add_argument("--test_helium", default="",\n                    help="pre-extracted helium .pt for the test clip")\n    ap.add_argument("--test_wav", default="", help="audio muxed into the test videos")\n    ap.add_argument("--test_out_dir", default="")\n    ap.add_argument("--test_emotion_id", type=int, default=-1,\n                    help="-1 = cycle all six emotions across tests")\n    ap.add_argument("--test_gain", type=float, default=1.2)\n    ap.add_argument("--test_seconds", type=float, default=6.0)\n    ap.add_argument("--motion_map", default="affine", choices=["affine", "clamp", "none"])\n    args = ap.parse_args()\n\n    repo = Path(args.repo_root).resolve()\n    src = repo / "src"\n    os.environ.setdefault("IMTALKER_DIR", str(src / "imtalker"))\n    os.environ.setdefault("DESIGN_A_DIR", str(src / "niloys"))\n    for p in (str(src / "imtalker"), str(src / "niloys"),\n              str(src / "niloys" / "emotion")):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n\n    import torch\n    import torch.nn.functional as F\n    from torch.utils.data import DataLoader, Dataset\n    from emotion_student import (EMOTION_NAMES, NUM_EMOTIONS, apply_emotion_lora,\n                                 load_base_student, parse_emotion_from_stem,\n                                 save_emotion_adapter)\n\n    torch.manual_seed(args.seed)\n    random.seed(args.seed)\n    device = "cuda"\n\n    model, cfg, pack = load_base_student(args.base_student)\n    model.emotion_conditioner.dropout = args.emotion_dropout\n    n_lora = apply_emotion_lora(model, rank=args.lora_rank, alpha=args.lora_alpha,\n                                dropout=args.lora_dropout)\n    mot_mean = model.motion_mean.detach().cpu().clone()\n    mot_std = model.motion_std.detach().cpu().clone()\n    model.to(device)\n    print("[train] base student: %s" % args.base_student, flush=True)\n    print("[train]   lookahead=%d attn_window=%d d_model=%d n_layers=%d"\n          % (cfg.lookahead, cfg.attn_window, cfg.d_model, cfg.n_layers), flush=True)\n    print("[train] LoRA wrapped %d layers (rank=%d alpha=%.0f dropout=%.2f)"\n          % (n_lora, args.lora_rank, args.lora_alpha, args.lora_dropout), flush=True)\n\n    L, W = cfg.lookahead, args.window_steps\n\n    # ---------------- dataset ----------------\n    def _load_clip(stem):\n        hp = os.path.join(args.helium_dir, stem + ".pt")\n        mp = os.path.join(args.motion_dir, stem + ".pt")\n        hpack = torch.load(hp, map_location="cpu", weights_only=False)\n        h = hpack["helium"] if isinstance(hpack, dict) else hpack        # [T,4096]\n        m = torch.load(mp, map_location="cpu", weights_only=False)\n        if isinstance(m, dict):\n            m = m.get("motion", m.get("motion_latents"))\n        return h.float(), m.float()\n\n    class WindowDataset(Dataset):\n        """Random windows: input helium[a+L : a+L+W] -> target motion[2a : 2(a+W)].\n\n        The lookahead shift L matches how the student was distilled: the output\n        for step s pairs with helium[s+L] (video lags audio by L*80 ms live).\n        prev_motion for window step j is motion[2(a+j)-2 : 2(a+j)] - the two\n        frames preceding the target pair, exactly what the streaming state holds.\n        """\n        def __init__(self, stems):\n            self.stems = list(stems)\n\n        def __len__(self):\n            return len(self.stems)\n\n        def __getitem__(self, i):\n            stem = self.stems[i]\n            eid = parse_emotion_from_stem(stem)\n            h, m = _load_clip(stem)\n            T = min(h.shape[0], m.shape[0] // 2)\n            lo, hi = args.warmup_steps, T - W - L\n            a = random.randint(lo, max(lo, hi))\n            inp = h[a + L: a + L + W]                                   # [W,4096] raw\n            tgt = m[2 * a: 2 * (a + W)].view(W, 2, -1)                  # [W,2,32] raw\n            if a > 0:\n                prev = m[2 * a - 2: 2 * a + 2 * (W - 1)]\n            else:\n                prev = torch.cat([torch.zeros(2, m.shape[1]), m[: 2 * (W - 1)]], 0)\n            tgt_n = (tgt - mot_mean) / mot_std\n            prev_n = ((prev.view(W, 2, -1) - mot_mean) / mot_std).reshape(W, -1)\n            return inp, prev_n.float(), tgt_n.float(), eid\n\n    # stratified split by emotion, val capped at 200 clips\n    by_emo = {e: [] for e in range(NUM_EMOTIONS)}\n    for f in sorted(glob.glob(os.path.join(args.helium_dir, "*.pt"))):\n        stem = Path(f).stem\n        e = parse_emotion_from_stem(stem)\n        if e is None:\n            continue\n        h, m = _load_clip(stem)\n        if min(h.shape[0], m.shape[0] // 2) - W - L > args.warmup_steps:\n            by_emo[e].append(stem)\n    rng = random.Random(args.seed)\n    train_stems, val_stems = [], []\n    for e in range(NUM_EMOTIONS):\n        ss = by_emo[e]\n        rng.shuffle(ss)\n        n_val = max(1, int(len(ss) * args.val_split)) if len(ss) > 4 else 1\n        val_stems += ss[:n_val]\n        train_stems += ss[n_val:]\n    val_stems = sorted(val_stems)[:200]\n    train_ds, val_ds = WindowDataset(train_stems), WindowDataset(val_stems)\n    print("[train] clips: %d train / %d val (per emotion: %s)"\n          % (len(train_ds), len(val_ds),\n             {EMOTION_NAMES[e]: len(by_emo[e]) for e in range(NUM_EMOTIONS)}),\n          flush=True)\n    assert len(train_ds) > 0, "no usable training clips - check the extraction step"\n    loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,\n                        num_workers=2, drop_last=True, pin_memory=True,\n                        persistent_workers=True)\n\n    # ---------------- optimiser (mirrors the old pipeline\'s two lr groups) ----------------\n    cond_params = [p for n, p in model.named_parameters()\n                   if "emotion_conditioner" in n and p.requires_grad]\n    lora_params = [p for n, p in model.named_parameters()\n                   if ("lora_A" in n or "lora_B" in n) and p.requires_grad]\n    print("[train] trainable params: %d (conditioner %d + LoRA %d)"\n          % (sum(p.numel() for p in cond_params + lora_params),\n             sum(p.numel() for p in cond_params),\n             sum(p.numel() for p in lora_params)), flush=True)\n    opt = torch.optim.AdamW(\n        [{"params": lora_params, "lr": args.lr},\n         {"params": cond_params, "lr": args.lr * args.cond_lr_mult}],\n        weight_decay=args.weight_decay)\n\n    def _meta(step, val=None):\n        return {"step": step, "lora_rank": args.lora_rank,\n                "lora_alpha": args.lora_alpha,\n                "base_student": os.path.basename(args.base_student),\n                "student_cfg": pack["cfg"], "emotion_names": EMOTION_NAMES,\n                "val": val or {}, "args": vars(args)}\n\n    @torch.no_grad()\n    def evaluate():\n        model.eval()\n        per_emo = {e: [] for e in range(NUM_EMOTIONS)}\n        for i in range(len(val_ds)):\n            inp, prev, tgt, eid = val_ds[i]\n            pred, _a, _r, _g = model.forward_with_emotion(\n                inp[None].to(device), prev[None].to(device),\n                torch.tensor([eid], device=device), train=False)\n            per_emo[eid].append(\n                F.mse_loss(pred.float(), tgt[None].to(device).float()).item())\n        model.train()\n        return {EMOTION_NAMES[e]: (sum(v) / len(v) if v else float("nan"))\n                for e, v in per_emo.items()}\n\n    @torch.no_grad()\n    def rollout_nmse():\n        """Free-running NMSE on one val clip per emotion - the number that\n        predicts live behaviour (the student\'s own tf_nmse metric style)."""\n        model.eval()\n        out = {}\n        for e in range(NUM_EMOTIONS):\n            stems_e = [s for s in val_ds.stems if parse_emotion_from_stem(s) == e]\n            if not stems_e:\n                continue\n            h, m = _load_clip(stems_e[0])\n            T = min(h.shape[0], m.shape[0] // 2) - L\n            st = model.init_state(1, device, torch.float32)\n            model.set_emotion(e)\n            outs = []\n            for t in range(T):\n                # step() -> [1,2,32]: BOTH motion frames of the 80 ms step (the\n                # state\'s prev_motion auto-feeds the free-running rollout)\n                outs.append(model.step(h[t + L: t + L + 1].to(device), st)[0])\n            pred = torch.cat(outs, 0).float().cpu()                 # [2T,32] norm\n            tgt = (m[: 2 * T] - mot_mean) / mot_std\n            out[EMOTION_NAMES[e]] = float(\n                ((pred - tgt) ** 2).mean() / (tgt ** 2).mean().clamp_min(1e-8))\n        model.train()\n        return out\n\n    @torch.no_grad()\n    def emotion_vectors():\n        model.eval()\n        # stack (not cat): each bias_for(...)[0, 0] is [d_model]; we want [NUM_EMOTIONS, d_model]\n        b = torch.stack([model.emotion_conditioner.bias_for(e, device, torch.float32)[0, 0]\n                         for e in range(NUM_EMOTIONS)], 0)\n        model.train()\n        norms = b.norm(dim=-1)\n        cos = F.cosine_similarity(b[None, :, :], b[:, None, :], dim=-1)\n        return norms, cos\n\n    # ---------------- periodic test videos ----------------\n    # Adapted from the old FM pipeline\'s EpochTestVideoCallback (every 10 epochs,\n    # ref image + audio + emotion -> video). New-architecture translation:\n    #   * ref image   -> the distilled renderer\'s baked-in identity (no ref needed)\n    #   * audio feats -> the PRE-EXTRACTED helium of a fixed test clip (the 7B\n    #                    trunk never runs inside the training process)\n    #   * Lightning epoch callback -> an epoch boundary in this step-based loop\n    # Rendering is no_grad + eval/train bracketed and reuses one lazily-loaded\n    # renderer; it cannot touch the optimizer or the emotion bias\n    # (forward_with_emotion saves/restores it; set_emotion writes the live buffer).\n    steps_per_epoch = max(1, -(-len(train_ds) // args.batch_size))\n    test_every = (args.test_every_n_epochs * steps_per_epoch\n                  if (args.test_every_n_epochs > 0 and args.test_helium) else 0)\n    test_state = {"R": None, "h": None, "n": 0}\n    if test_every > 0:\n        import render_emotion_tests as ret\n        hpack = torch.load(args.test_helium, map_location="cpu", weights_only=False)\n        test_state["h"] = (hpack["helium"] if isinstance(hpack, dict) else hpack).float()\n        print("[test] periodic test videos: every %d epochs (%d steps), %.0f s clip -> %s"\n              % (args.test_every_n_epochs, test_every, args.test_seconds,\n                 args.test_out_dir), flush=True)\n\n    @torch.no_grad()\n    def run_test_video(epoch):\n        if test_state["h"] is None:\n            return\n        if test_state["R"] is None:\n            test_state["R"], _rp = ret.load_renderer(args.repo_root, device)\n        eid = (test_state["n"] % NUM_EMOTIONS\n               if args.test_emotion_id < 0 else args.test_emotion_id)\n        os.makedirs(args.test_out_dir, exist_ok=True)\n        out = os.path.join(args.test_out_dir, "epoch_%04d_%s.mp4"\n                           % (epoch, EMOTION_NAMES[eid]))\n        model.eval()\n        ret.render_motion_video(model, test_state["R"], test_state["h"], cfg, eid,\n                                args.test_wav or None, out, device,\n                                gain=args.test_gain, max_seconds=args.test_seconds,\n                                motion_map=args.motion_map)\n        model.train()\n        test_state["n"] += 1\n        print("[test] epoch %d -> %s" % (epoch, out), flush=True)\n\n    run_test_video(0)   # baseline: the untrained adapter on the test clip\n\n    # ---------------- loop ----------------\n    os.makedirs(args.out_dir, exist_ok=True)\n    model.train()\n    step, ema, t0 = 0, None, time.time()\n    it = iter(loader)\n    while step < args.max_steps:\n        try:\n            inp, prev, tgt, eid = next(it)\n        except StopIteration:\n            it = iter(loader)\n            inp, prev, tgt, eid = next(it)\n        inp = inp.to(device, non_blocking=True)\n        prev = prev.to(device, non_blocking=True)\n        tgt = tgt.to(device, non_blocking=True)\n        eid = eid.to(device, non_blocking=True)\n        pred, _a, _r, _g = model.forward_with_emotion(inp, prev, eid, train=True)\n        loss = F.mse_loss(pred, tgt)\n        opt.zero_grad(set_to_none=True)\n        loss.backward()\n        gn = torch.nn.utils.clip_grad_norm_(cond_params + lora_params, args.grad_clip)\n        opt.step()\n        step += 1\n        ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()\n        if step % 50 == 0:\n            print("[train] step %6d/%d  loss %.5f  ema %.5f  |g| %.3f  %.1f it/s"\n                  % (step, args.max_steps, loss.item(), ema, float(gn),\n                     step / (time.time() - t0)), flush=True)\n        if test_every > 0 and (step % test_every == 0 or step == args.max_steps):\n            run_test_video(step // steps_per_epoch)\n        if step % args.val_every == 0 or step == args.max_steps:\n            mse = evaluate()\n            nmse = rollout_nmse()\n            norms, cos = emotion_vectors()\n            off = cos[~torch.eye(NUM_EMOTIONS, dtype=torch.bool, device=cos.device)]\n            print("[val] step %d  teacher-forced MSE per emotion: %s"\n                  % (step, {k: round(v, 5) for k, v in mse.items()}), flush=True)\n            print("[val]   free-running NMSE: %s"\n                  % {k: round(v, 4) for k, v in nmse.items()}, flush=True)\n            print("[val]   emotion bias norms: %s  pairwise cos: min %.3f max %.3f"\n                  % ([round(float(x), 3) for x in norms],\n                     float(off.min()), float(off.max())), flush=True)\n            save_emotion_adapter(model, os.path.join(args.out_dir, "last.pt"),\n                                 _meta(step, {"tf_mse": mse, "rollout_nmse": nmse}))\n        if step % args.save_every == 0:\n            save_emotion_adapter(model,\n                                 os.path.join(args.out_dir, "step_%06d.pt" % step),\n                                 _meta(step))\n    print("TRAINING DONE: %d steps, adapter -> %s"\n          % (step, os.path.join(args.out_dir, "last.pt")), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
path = os.path.join(REPO_DIR, "src", "niloys", "emotion", "train_emotion.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))

In [ ]:
TRAIN_LOG = os.path.join(OUT_DIR, "train.log")
TEST_VIDEO_DIR = os.path.join(OUT_DIR, "test_videos")
# resolve the fixed periodic-test clip: its helium was extracted in section 3
if not TEST_VIDEO_CLIP:
    if TEST_CLIP:
        TEST_VIDEO_CLIP = TEST_CLIP
    else:
        cands = sorted(glob.glob(os.path.join(HELIUM_DIR, "*.pt")))
        assert cands, "run the extraction first"
        TEST_VIDEO_CLIP = Path(cands[0]).stem
TEST_HELIUM = os.path.join(HELIUM_DIR, TEST_VIDEO_CLIP + ".pt")
assert os.path.exists(TEST_HELIUM), "test helium missing: %s" % TEST_HELIUM
TEST_WAV = ""
for ext in ("wav", "mp3", "flac", "ogg", "m4a"):
    c = os.path.join(RAW_AUDIO_DIR, TEST_VIDEO_CLIP + "." + ext)
    if os.path.exists(c):
        TEST_WAV = c
        break
print("periodic test clip:", TEST_VIDEO_CLIP, " wav:", TEST_WAV or "(none)")
sh("pkill -f train_emotion.py; sleep 1", check=False)
cmd = ("%s -u src/niloys/emotion/train_emotion.py "
       "--repo_root . --helium_dir %s --motion_dir %s --out_dir %s --base_student %s "
       "--lora_rank %d --lora_alpha %s --lora_dropout %s --lr %s --cond_lr_mult %s "
       "--batch_size %d --window_steps %d --emotion_dropout %s --warmup_steps %d "
       "--max_steps %d --val_every %d --save_every %d "
       "--test_every_n_epochs %d --test_helium %s --test_wav %s --test_out_dir %s "
       "--test_emotion_id %d --test_gain 1.2 --test_seconds %s --motion_map %s"
       % (shlex.quote(PY),
          shlex.quote(HELIUM_DIR),
          shlex.quote(os.path.join(EMOTION_DATASET_DIR, "motion")),
          shlex.quote(CKPT_DIR), shlex.quote(BASE_STUDENT),
          LORA_RANK, LORA_ALPHA, LORA_DROPOUT, LR, COND_LR_MULT,
          BATCH_SIZE, WINDOW_STEPS, EMOTION_DROPOUT, EXTRACT_WARMUP_STEPS,
          MAX_STEPS, VAL_EVERY, SAVE_EVERY,
          TEST_EVERY_N_EPOCHS, shlex.quote(TEST_HELIUM), shlex.quote(TEST_WAV),
          shlex.quote(TEST_VIDEO_DIR),
          TEST_EMOTION_ID, TEST_VIDEO_SECONDS, shlex.quote(MOTION_MAP)))
sh("nohup %s > %s 2>&1 & sleep 3; pgrep -af train_emotion.py | head -2"
   % (cmd, shlex.quote(TRAIN_LOG)))

In [ ]:
# follow the training (re-runnable); returns when the process exits or TRAINING DONE appears
t0 = time.time()
while True:
    alive = subprocess.run(["pgrep", "-f", "train_emotion.py"],
                           capture_output=True).returncode == 0
    tail = subprocess.run(["bash", "-c", "tail -n 3 %s 2>/dev/null" % TRAIN_LOG],
                          capture_output=True, text=True).stdout
    last = tail.strip().splitlines()[-1] if tail.strip() else ""
    print("\r%7.0fs  %s" % (time.time() - t0, last[-160:]), end="", flush=True)
    if not alive or "TRAINING DONE" in tail:
        break
    time.sleep(20)
print()
sh("grep -aE 'TRAINING DONE|\\[val\\]|\\[test\\]' %s | tail -40" % TRAIN_LOG, check=False)
sh("ls -lh %s 2>/dev/null | tail -10" % TEST_VIDEO_DIR, check=False)

## 5 · Adapter validation

Loads `checkpoints/last.pt`, merges the LoRA into the base student on CPU, and prints the
emotion-bias geometry. What you want to see: **non-zero, distinct bias vectors** (pairwise
cosines well below 1 for most pairs) and a sensible validation record. If every bias is
near zero or all six are nearly identical, training did not take — check the log.

In [ ]:
CHECK = r"""
import sys, torch
sys.path.insert(0, 'src/imtalker'); sys.path.insert(0, 'src/niloys'); sys.path.insert(0, 'src/niloys/emotion')
from emotion_student import EMOTION_NAMES, NUM_EMOTIONS, build_emotion_student
model, cfg, meta = build_emotion_student(r'BASE_STUDENT_X', r'ADAPTER_X', 'cpu')
print('adapter meta :', {k: meta.get(k) for k in ('step', 'lora_rank', 'lora_alpha', 'base_student')})
print('validation   :', meta.get('val'))
import torch.nn.functional as F
b = torch.stack([model.emotion_conditioner.bias_for(e, 'cpu', torch.float32)[0, 0]
                 for e in range(NUM_EMOTIONS)], 0)   # [NUM_EMOTIONS, d_model]
print('bias norms   :', {EMOTION_NAMES[e]: round(float(b[e].norm()), 3) for e in range(NUM_EMOTIONS)})
cos = F.cosine_similarity(b[None], b[:, None], dim=-1)
print('pairwise cos :')
for e in range(NUM_EMOTIONS):
    print('  %-9s %s' % (EMOTION_NAMES[e], ' '.join('%5.2f' % float(cos[e, j]) for j in range(NUM_EMOTIONS))))
assert float(b.norm()) > 0, 'all emotion biases are zero - training did not run'
"""
check = (CHECK
         .replace("BASE_STUDENT_X", BASE_STUDENT)
         .replace("ADAPTER_X", os.path.join(CKPT_DIR, "last.pt")))
open(os.path.join(REPO_DIR, "_check_adapter.py"), "w").write(check)
sh("%s _check_adapter.py && rm _check_adapter.py" % shlex.quote(PY))

## 6 · Test videos for all six emotions

Free-running rollout of the emotion student on one held-out clip — exactly the code path
the live server takes (no teacher forcing, same motion gain 1.2) — rendered with the
distilled single-identity renderer and muxed with the clip's audio. Watch them side by
side: lip-sync should be identical across emotions (the Helium path is frozen), while
brows/eyes/expression should follow the emotion id. The printed
`mean |motion − neutral|` row per emotion is the quantitative check that the bias does
something.

**The white-video fix (`MOTION_MAP`).** The distilled renderer was trained on ONE
identity's motion range (Robert); dataset motion sits outside it, which saturates the
FiLM-conditioned residual generator into white frames. `affine` (default) standardizes
the clip's motion per dimension onto the renderer's stats — within-clip dynamics (the
emotion) are preserved, only the identity-level pose/scale is remapped; a safety clamp
at renderer mean ± 4 σ follows. The run also prints the sigma-range diagnostics per
emotion, saves a `*_first_frame.png` per video for a quick look without downloading the
mp4, and renders two **ground-truth diagnostic videos** (`gt_none_*`, `gt_affine_*`) from
the dataset's own motion: if `gt_none` is white but `gt_affine` shows a talking face,
the cause is conclusively the motion range, not the student or the extraction.

In [ ]:
import os
CONTENT = '"""Render one clip through the emotion student for every emotion id, with the\ndistilled single-identity renderer - the offline equivalent of the old\npipeline\'s test_emotions.py. Free-running rollout (no teacher forcing), the\nsame code path the live server takes, so these videos predict live behaviour.\n\nMotion-range handling (the white-video fix): the distilled renderer predicts a\nresidual on a baked-in base frame, conditioned via FiLM on\n(motion - R.motion_mean) / R.motion_std. It was distilled on ONE identity\'s\nmotion distribution (Robert); emotional datasets sit far outside it, which\nsaturates the FiLM conditioning and clamps every frame to white. The default\n--motion_map affine standardizes the clip\'s motion per dimension onto the\nrenderer stats (within-clip dynamics - the emotion signal - are preserved;\nonly the identity-level pose/scale is remapped onto Robert), followed by a\nsafety clamp at renderer mean +/- 4 sigma.\n"""\nimport argparse, os, subprocess, sys\nfrom pathlib import Path\n\nFPS = 25                      # 2 motion frames per 80 ms step\n\n\ndef _mux_audio(tmp, wav_path, out_path):\n    if wav_path:\n        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,\n                        "-i", wav_path, "-c:v", "copy", "-c:a", "aac",\n                        "-shortest", out_path], check=True)\n        os.remove(tmp)\n    else:\n        os.replace(tmp, out_path)\n\n\ndef find_audio(raw_audio_dir, stem):\n    for ext in ("wav", "mp3", "flac", "ogg", "m4a"):\n        c = os.path.join(raw_audio_dir, stem + "." + ext)\n        if os.path.exists(c):\n            return c\n    return None\n\n\ndef load_renderer(repo_root, device):\n    import torch\n    from student.renderer import build as build_renderer\n    rpack = torch.load(Path(repo_root) / "weights" / "renderer_narrow.pt",\n                       map_location="cpu", weights_only=False)\n    R = build_renderer(**rpack["cfg"]).to(device).float().eval()\n    R.load_state_dict(rpack["model"])\n    for p in R.parameters():\n        p.requires_grad_(False)\n    return R, rpack\n\n\ndef rollout_motion(model, helium, cfg, emotion_id, device, gain=1.2,\n                   max_seconds=12.0):\n    """Free-running rollout -> [2T, 32] RAW (denormalized) motion on CPU."""\n    import torch\n    T = min(helium.shape[0] - cfg.lookahead, int(max_seconds * 12.5))\n    model.set_emotion(emotion_id)\n    st = model.init_state(1, device, torch.float32)\n    outs = []\n    with torch.no_grad():\n        for t in range(T):\n            o = model.step(helium[t + cfg.lookahead: t + cfg.lookahead + 1]\n                           .to(device), st)\n            outs.append(o[0])\n    pred_n = torch.stack(outs, 0)                          # [T,2,32] normalized\n    if gain != 1.0:\n        mu = pred_n.mean(dim=0, keepdim=True)\n        pred_n = mu + (pred_n - mu) * gain\n    return model.denorm_motion(pred_n).reshape(-1, cfg.motion_dim).float().cpu()\n\n\ndef map_motion_for_renderer(motion, R, mode="affine", clamp_sigmas=4.0):\n    """Bring raw [N,32] motion into the distilled renderer\'s training range.\n\n    \'affine\': standardize the clip per dimension onto the renderer stats.\n    \'clamp\' : only clip to renderer mean +/- clamp_sigmas * std.\n    \'none\'  : leave the motion untouched (diagnostic).\n    Returns (mapped_motion, stats_string).\n    """\n    import torch\n    mean, std = R.motion_mean.cpu(), R.motion_std.cpu().clamp_min(1e-6)\n    lo, hi = mean - clamp_sigmas * std, mean + clamp_sigmas * std\n    raw = motion.clone()\n    if mode == "affine":\n        cm, cs = motion.mean(0), motion.std(0).clamp_min(1e-6)\n        motion = (motion - cm) / cs * std + mean\n    clipped = ((motion < lo) | (motion > hi)).float().mean().item()\n    motion = torch.maximum(torch.minimum(motion, hi), lo)\n    stats = ("raw range [%+.3f, %+.3f]  renderer sigma range [%+.1f, %+.1f]  "\n             "clamped %.1f%% of values"\n             % (raw.min().item(), raw.max().item(),\n                ((raw - mean) / std).min().item(),\n                ((raw - mean) / std).max().item(), 100 * clipped))\n    return motion, stats\n\n\ndef render_video_frames(R, motion, device, out_prefix=None):\n    """raw [N,32] motion -> uint8 BGR frames [N,H,W,3]; dumps the first frame."""\n    import numpy as np\n    import torch\n    frames = []\n    with torch.no_grad():\n        for i in range(0, motion.shape[0], 16):\n            img = R(motion[i: i + 16].to(device))          # [B,3,H,W] in [0,1]\n            frames.append((img.permute(0, 2, 3, 1) * 255).byte().cpu().numpy())\n    frames = np.concatenate(frames, 0)[:, :, :, ::-1]      # RGB -> BGR\n    if out_prefix:\n        import cv2\n        cv2.imwrite(out_prefix + "_first_frame.png", frames[0])\n    return frames\n\n\ndef write_video(frames, wav_path, out_path, fps=FPS):\n    import cv2\n    H, W = frames.shape[1:3]\n    tmp = out_path + ".silent.mp4"\n    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))\n    for fr in frames:\n        vw.write(fr)\n    vw.release()\n    _mux_audio(tmp, wav_path, out_path)\n    return out_path\n\n\ndef render_motion_video(model, R, helium, cfg, emotion_id, wav_path, out_path,\n                        device, gain=1.2, max_seconds=12.0, motion_map="affine"):\n    """rollout -> range map -> frames -> mp4 (audio muxed).\n\n    Shared by this CLI and by the periodic test inside train_emotion.py.\n    Returns (raw_motion, stats_string, n_frames).\n    """\n    motion = rollout_motion(model, helium, cfg, emotion_id, device,\n                            gain=gain, max_seconds=max_seconds)\n    mapped, stats = map_motion_for_renderer(motion, R, motion_map)\n    frames = render_video_frames(R, mapped, device,\n                                 out_prefix=os.path.splitext(out_path)[0])\n    write_video(frames, wav_path, out_path)\n    return motion, stats, frames.shape[0]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--repo_root", required=True)\n    ap.add_argument("--base_student", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--helium_dir", required=True)\n    ap.add_argument("--raw_audio_dir", required=True)\n    ap.add_argument("--clip_stem", required=True)\n    ap.add_argument("--out_dir", required=True)\n    ap.add_argument("--motion_dir", default="",\n                    help="dataset motion dir -> also renders the ground-truth "\n                         "motion diagnostic videos (gt_none/gt_affine)")\n    ap.add_argument("--gain", type=float, default=1.2,\n                    help="same motion gain the live server applies")\n    ap.add_argument("--max_seconds", type=float, default=12.0)\n    ap.add_argument("--motion_map", default="affine",\n                    choices=["affine", "clamp", "none"])\n    args = ap.parse_args()\n\n    repo = Path(args.repo_root).resolve()\n    src = repo / "src"\n    os.environ.setdefault("DESIGN_A_DIR", str(src / "niloys"))\n    for p in (str(src / "niloys"), str(src / "niloys" / "emotion")):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n\n    import torch\n    from emotion_student import EMOTION_NAMES, NUM_EMOTIONS, build_emotion_student\n\n    device = "cuda"\n    model, cfg, meta = build_emotion_student(args.base_student, args.adapter, device)\n    print("[render] student + adapter loaded (trained step %s)"\n          % meta.get("step"), flush=True)\n\n    R, rpack = load_renderer(repo, device)\n    print("[render] distilled renderer loaded (val_PSNR=%.2f dB)"\n          % rpack.get("val_psnr", 0.0), flush=True)\n\n    # ---- range diagnostics: student stats vs the renderer\'s expectations ----\n    rmean, rstd = R.motion_mean.cpu(), R.motion_std.cpu()\n    smean, sstd = model.motion_mean.cpu(), model.motion_std.cpu()\n    smean, sstd = model.motion_mean.cpu(), model.motion_std.cpu()\n    print("[render] student  motion stats: mean |%.3f|  std %.3f"\n          % (float(smean.abs().mean()), float(sstd.mean())), flush=True)\n    print("[render] renderer motion stats: mean |%.3f|  std %.3f"\n          % (float(rmean.abs().mean()), float(rstd.mean())), flush=True)\n\n    hpack = torch.load(os.path.join(args.helium_dir, args.clip_stem + ".pt"),\n                       map_location="cpu", weights_only=False)\n    h = (hpack["helium"] if isinstance(hpack, dict) else hpack).float()\n    T = min(h.shape[0] - cfg.lookahead, int(args.max_seconds * 12.5))\n    print("[render] clip %s: %d steps (%.1f s)"\n          % (args.clip_stem, T, T / 12.5), flush=True)\n\n    wav_path = find_audio(args.raw_audio_dir, args.clip_stem)\n    os.makedirs(args.out_dir, exist_ok=True)\n\n    # ---- ground-truth diagnostic: the DATASET\'s motion through the renderer.\n    # If gt_none is white but gt_affine shows a talking face, the white videos\n    # are a motion-range problem (OOD w.r.t. Robert), independent of the\n    # student, the extraction and the training.\n    if args.motion_dir:\n        mp = os.path.join(args.motion_dir, args.clip_stem + ".pt")\n        if os.path.exists(mp):\n            m = torch.load(mp, map_location="cpu", weights_only=False)\n            if isinstance(m, dict):\n                m = m.get("motion", m.get("motion_latents"))\n            gt = m.float()[: 2 * T]\n            sig = (gt - rmean) / rstd.clamp_min(1e-6)\n            print("[render] GT motion range [%+.3f, %+.3f]  sigma range [%+.1f, %+.1f]"\n                  % (float(gt.min()), float(gt.max()),\n                     float(sig.min()), float(sig.max())), flush=True)\n            for mode in ("none", "affine"):\n                m2, _st = map_motion_for_renderer(gt.clone(), R, mode)\n                frames = render_video_frames(R, m2, device,\n                                             out_prefix=os.path.join(\n                                                 args.out_dir, "gt_%s" % mode))\n                out = os.path.join(args.out_dir, "gt_%s_%s.mp4"\n                                   % (mode, args.clip_stem))\n                write_video(frames, wav_path, out)\n                print("[render] GT %s -> %s" % (mode, out), flush=True)\n\n    motions = {}\n    for eid in range(NUM_EMOTIONS):\n        out_path = os.path.join(args.out_dir, "%d_%s.mp4" % (eid, EMOTION_NAMES[eid]))\n        motion, stats, nfr = render_motion_video(\n            model, R, h, cfg, eid, wav_path, out_path, device,\n            gain=args.gain, max_seconds=args.max_seconds,\n            motion_map=args.motion_map)\n        motions[eid] = motion\n        print("[render] %-8s -> %s (%d frames)"\n              % (EMOTION_NAMES[eid], out_path, nfr), flush=True)\n        print("[render]   motion[%s]  %s" % (EMOTION_NAMES[eid], stats), flush=True)\n\n    ref = motions[0]\n    print("[render] mean |motion - neutral| per emotion (raw units):", flush=True)\n    for eid in range(1, NUM_EMOTIONS):\n        d = (motions[eid] - ref).abs().mean().item()\n        print("[render]   %-9s %.4f" % (EMOTION_NAMES[eid], d), flush=True)\n    print("RENDER DONE -> %s" % args.out_dir, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n\n'
path = os.path.join(REPO_DIR, "src", "niloys", "emotion", "render_emotion_tests.py")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(CONTENT)
print("wrote", path, "(%d chars)" % len(CONTENT))

In [ ]:
if not TEST_CLIP:
    cands = sorted(glob.glob(os.path.join(HELIUM_DIR, "*.pt")))
    assert cands, "run the extraction first"
    TEST_CLIP = Path(cands[0]).stem
print("test clip:", TEST_CLIP)
sh("%s -u src/niloys/emotion/render_emotion_tests.py "
   "--repo_root . --base_student %s --adapter %s --helium_dir %s --raw_audio_dir %s "
   "--clip_stem %s --out_dir %s --motion_dir %s --motion_map %s "
   "--gain 1.2 --max_seconds %s"
   % (shlex.quote(PY), shlex.quote(BASE_STUDENT),
      shlex.quote(os.path.join(CKPT_DIR, "last.pt")),
      shlex.quote(HELIUM_DIR), shlex.quote(RAW_AUDIO_DIR),
      shlex.quote(TEST_CLIP), shlex.quote(VIDEO_DIR),
      shlex.quote(os.path.join(EMOTION_DATASET_DIR, "motion")),
      shlex.quote(MOTION_MAP), TEST_MAX_SECONDS))
sh("ls -lh %s" % VIDEO_DIR, check=False)

## 7 · Outputs, usage, troubleshooting

**The adapter** — `checkpoints/last.pt` (plus `step_NNNNNN.pt` snapshots). Point
`emotion_live_streaming.ipynb` at it (`EMOTION_LORA_CKPT`) to run the live server with
emotion control.

**Optional: push the adapter to Hugging Face** (so the live notebook can fetch it on any
pod):

```python
from huggingface_hub import HfApi
HfApi().upload_file(path_or_fileobj=os.path.join(CKPT_DIR, "last.pt"),
                    path_in_repo="last.pt", repo_id="YOUR_USER/student_emotion_lora_v1",
                    repo_type="model")
```

then set `EMOTION_HF_REPO = "YOUR_USER/student_emotion_lora_v1"` in the live notebook.

| symptom | fix |
|---|---|
| validation cell: "with raw audio" ≪ motion clips | the wav stems must match the motion stems exactly (same basename) |
| `no clip has ... a parseable emotion label` | your stems do not contain an emotion word; extend `EMOTION_MAP` in `src/niloys/emotion/emotion_student.py` |
| verify cell: cos < 0.5 or std ratio far from 1 | extraction mismatch — make sure `EXTRACT_INT4` matches `.gpu_probe.env` (the live server) and the voice/persona prompt matches `run_x86.sh` |
| extraction `CUDA out of memory` | stop the live server first (`pkill -f imtalker_try19_student_live.py`); the trunk needs ~14 GB (int4) |
| training loss NaN early | lower `LR` to `5e-5` |
| all six test videos identical | the bias is not being used — check `[val] emotion bias norms` (should be ≫ 0) and, live, that `EMOTION_ID`/the control file reach the server |
| rollout NMSE ≈ 1.0 for every emotion | training did not converge — run more steps or re-check the extraction verification |